<div style="background-color:#EF4444; padding:18px 22px; border-left:6px solid #B91C1C; border-radius:8px;">

# **KDE**

</div>

In [2]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from scipy.stats import gaussian_kde

class KDEPairEnsembleClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, feature_cols, pairs, grid_size=200):
        self.feature_cols = feature_cols
        self.pairs = pairs
        self.grid_size = grid_size

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        c0, c1 = self.classes_

        self.grid_store_ = {}
        for xi, xj in self.pairs:
            ix = self.feature_cols.index(xi)
            jx = self.feature_cols.index(xj)

            x0 = X[y == c0][:, [ix, jx]]
            x1 = X[y == c1][:, [ix, jx]]

            kde0 = gaussian_kde(x0.T)
            kde1 = gaussian_kde(x1.T)

            x_min, y_min = X[:, [ix, jx]].min(axis=0)
            x_max, y_max = X[:, [ix, jx]].max(axis=0)
            x_edges = np.linspace(x_min, x_max, self.grid_size)
            y_edges = np.linspace(y_min, y_max, self.grid_size)
            xx, yy = np.meshgrid(x_edges, y_edges)
            grid_pts = np.vstack([xx.ravel(), yy.ravel()])

            f0 = kde0(grid_pts).reshape(xx.shape)
            f1 = kde1(grid_pts).reshape(xx.shape)

            p1 = f1 / (f0 + f1 + 1e-300)
            p0 = 1 - p1

            self.grid_store_[(xi, xj)] = {
                "x_edges": x_edges, "y_edges": y_edges, "p0": p0, "p1": p1
            }

        return self

    def predict(self, X):
        X = np.asarray(X)
        p0_sum = np.zeros(len(X))
        p1_sum = np.zeros(len(X))

        for (xi, xj), store in self.grid_store_.items():
            ix = self.feature_cols.index(xi)
            jx = self.feature_cols.index(xj)

            xvals = X[:, ix]
            yvals = X[:, jx]

            ixg = np.clip(np.searchsorted(store["x_edges"], xvals) - 1, 0, self.grid_size - 1)
            iyg = np.clip(np.searchsorted(store["y_edges"], yvals) - 1, 0, self.grid_size - 1)

            p0_sum += store["p0"][iyg, ixg]
            p1_sum += store["p1"][iyg, ixg]

        return np.where(p1_sum > p0_sum, self.classes_[1], self.classes_[0])

In [6]:
import joblib
import pandas as pd

# Charger le modèle
pipeline = joblib.load("kde_ensemble_model.joblib")

# Charger les nouvelles données
DATA_PATH =r"C:\Users\victo\OneDrive\Bureau\OCP12\P12 billets_test.csv"
df = pd.read_csv(DATA_PATH, sep=",")

feature_cols = ["length", "margin_low", "margin_up"]

# Créer la colonne prediction
df["prediction"] = pd.NA

# Identifier les lignes sans NA
mask = df[feature_cols].notna().all(axis=1)

# Prédire uniquement sur les lignes valides
df.loc[mask, "prediction"] = pipeline.predict(
    df.loc[mask, feature_cols]
)

display(df[feature_cols + ["prediction"]])

,length,margin_low,margin_up,prediction
0,113.19,4.39,3.09,True
1,111.82,5.27,3.16,False
2,113.39,3.81,3.24,True
3,111.10,5.57,3.30,False
4,111.87,5.00,3.07,False


<div style="background-color:#4A90E2; padding:18px 22px; border-left:6px solid #357ABD; border-radius:8px;">

# **LOGISTIC REGRESSION**

</div>

In [10]:

import joblib
import pandas as pd


# ============================================================
# 1. Charger le modèle
# ============================================================

MODEL_PATH = "logistic_regression_4_couples.joblib"

pipeline = joblib.load(MODEL_PATH)

print("✅ Modèle LR chargé")


# ============================================================
# 2. Charger les nouvelles données
# ============================================================

df = pd.read_csv(DATA_PATH, sep=",")


# ============================================================
# 3. Variables utilisées par le modèle
# ============================================================

feature_cols = [
    "length",
    "margin_low",
    "margin_up",
    "height_right",
    "height_left"
]

X_new = df[feature_cols]


# ============================================================
# 4. Gérer les NA
# ============================================================

# On crée la colonne de prédiction
df["prediction_lr"] = pd.NA

# On ne prédit que les lignes complètes
mask = X_new.notna().all(axis=1)


# ============================================================
# 5. Prédictions
# ============================================================

df.loc[mask, "prediction_lr"] = pipeline.predict(
    X_new.loc[mask]
)


# ============================================================
# 6. Afficher les résultats
# ============================================================

display(
    df[feature_cols + ["prediction_lr"]]
)


✅ Modèle LR chargé


,length,margin_low,margin_up,height_right,height_left,prediction_lr
0,113.19,4.39,3.09,103.73,103.95,True
1,111.82,5.27,3.16,104.03,104.17,False
2,113.39,3.81,3.24,103.75,103.80,True
3,111.10,5.57,3.30,103.99,104.08,False
4,111.87,5.00,3.07,104.37,104.34,False
